# RQ 5 · Custom CNN **vs.** Transfer‑Learning ResNet‑18

This experiment addresses **Research Question 5** of our project:

> *How does a purpose‑built lightweight CNN compare with a transfer‑learned, ImageNet‑pre‑trained backbone (ResNet‑18) for multi‑class brain‑tumor MRI classification?*

Three models are evaluated side‑by‑side:

| Model | Abbrev. | Pre‑training | Input size | # Params | Dropout | Notes |
|-------|---------|-------------|------------|----------|---------|-------|
| SimpleCNN | **SCNN** | None | 160 × 160 | *≈* 0.91 M | 0.0 | Patched first conv to accept 3‑channel tensors |
| **LpCNN** | **LpCNN** | None | 32 × 32 | *≈* 2.1 M | 0.0 | Uses learnable polyphase (Lp‑conv) filters |
| ResNet‑18 | **RN18‑TL** | ImageNet‑1K | 160 × 160 | 11.7 M | 0.0 | Last layer replaced; backbone unfrozen after epoch 1 |

All three operate on the **same 3‑channel transform**, ensuring channel‑mismatch errors are eliminated and results are directly comparable.


## 1 Dataset & Transforms

* **Dataset root**: `../cleaned_data`  
* **Classes** (4): `NoTumor, Pituitary, Meningioma, Glioma`  
* **Split**: Author‑provided *Training* / *Testing* directories  
* **Input modalities**: single‑channel T1‑weighted MR slices  
* **Transform pipeline**

In [33]:

import time, os, copy, random, sys
from collections import defaultdict

import numpy as np
import pandas as pd
import torch, torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report

# Project modules
sys.path.append(os.path.abspath(".."))
from src.dataset import BrainTumorDataset
from src.transforms_config import create_transform
from src.models import SimpleCNN
print('Torch', torch.__version__)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device →', DEVICE)


Torch 2.7.0+cpu
Device → cpu


In [34]:

DATA_ROOT    = '../cleaned_data'   # path to dataset
INPUT_SIZE   = 160
BATCH_SIZE   = 16
N_EPOCHS     = 50
LR           = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT      = 0.0
NUM_CLASSES  = 4

# 3‑channel transform (one_ch_to_three_ch=True)
TRANSFORM = create_transform(
    image_size=(INPUT_SIZE, INPUT_SIZE),
    augment=False,
    adv_aug=False,
    strong_aug=False,
    one_ch_to_three_ch=True
)


In [35]:

train_set = BrainTumorDataset(DATA_ROOT, subset='Training', transform=TRANSFORM, model_variant='simple')
val_set   = BrainTumorDataset(DATA_ROOT, subset='Testing',  transform=TRANSFORM, model_variant='simple')

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train samples: {len(train_set)} | Val samples: {len(val_set)}')


Train samples: 5442 | Val samples: 1284


In [36]:

def train_one_epoch(model, loader, criterion, optimizer):
    model.train(); running_loss=0.0; running_corrects=0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward(); optimizer.step()
        running_loss += loss.item()*x.size(0)
        running_corrects += (logits.argmax(1)==y).sum().item()
    return running_loss/len(loader.dataset), running_corrects/len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval(); loss=0.0; corr=0; y_true=[]; y_pred=[]
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x); loss += criterion(logits, y).item()*x.size(0)
        preds = logits.argmax(1)
        corr += (preds==y).sum().item()
        y_true.extend(y.cpu().numpy()); y_pred.extend(preds.cpu().numpy())
    acc = corr/len(loader.dataset); f1 = f1_score(y_true, y_pred, average='macro')
    return loss/len(loader.dataset), acc, f1, y_true, y_pred


## 3 Model A – SimpleCNN (SCNN)

* **Architecture**: 3 × `Conv → ReLU → MaxPool` blocks + 2 FC layers  
* **Patch**: First convolution modified from `1 → 3` input channels  
* **Optimizer**: Adam `lr = 1e‑3`, `weight_decay = 1e‑4`  
* **Training**: 50 epochs, no LR schedule  

> **Best SCNN accuracy:** **`{best_acc:.2%}`** (F1 =`93.07`)  

*(Insert the printed value from the log.)*


In [37]:

# --- SimpleCNN (patched for 3‑channel input) ---
custom_model = SimpleCNN(
    num_classes=NUM_CLASSES,
    input_size=INPUT_SIZE,
    dropout=DROPOUT
).to(DEVICE)

# If the first conv expects 1 channel, swap it for 3‑channel version
first_conv = None
for name, module in custom_model.named_modules():
    if isinstance(module, nn.Conv2d):
        first_conv = module
        break
if first_conv is not None and first_conv.in_channels == 1:
    print('Patching SimpleCNN first conv: 1‑→3 channels')
    new_conv = nn.Conv2d(
        in_channels=3,
        out_channels=first_conv.out_channels,
        kernel_size=first_conv.kernel_size,
        stride=first_conv.stride,
        padding=first_conv.padding,
        bias=first_conv.bias is not None
    )
    custom_model.features[0] = new_conv  # assumes Sequential
else:
    print('SimpleCNN already compatible.')

crit = nn.CrossEntropyLoss()
opt  = optim.Adam(custom_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_acc = 0.0
for ep in range(N_EPOCHS):
    train_one_epoch(custom_model, train_loader, crit, opt)
    _, v_acc, v_f1, _, _ = evaluate(custom_model, val_loader, crit)
    print(f'[Custom] {ep+1:02d}/{N_EPOCHS}  Val acc {v_acc*100:5.2f}%  F1 {v_f1:.3f}')
    best_acc = max(best_acc, v_acc)
print(f'Best CustomCNN accuracy: {best_acc*100:.2f}%')


Patching SimpleCNN first conv: 1‑→3 channels


[Custom] 01/50  Val acc 76.95%  F1 0.727
[Custom] 02/50  Val acc 81.31%  F1 0.810
[Custom] 03/50  Val acc 88.16%  F1 0.875
[Custom] 04/50  Val acc 89.33%  F1 0.891
[Custom] 05/50  Val acc 91.74%  F1 0.914
[Custom] 06/50  Val acc 91.51%  F1 0.913
[Custom] 07/50  Val acc 92.60%  F1 0.924
[Custom] 08/50  Val acc 92.06%  F1 0.917
[Custom] 09/50  Val acc 90.50%  F1 0.901
[Custom] 10/50  Val acc 91.12%  F1 0.911
[Custom] 11/50  Val acc 92.68%  F1 0.925
[Custom] 12/50  Val acc 92.76%  F1 0.926
[Custom] 13/50  Val acc 91.20%  F1 0.912
[Custom] 14/50  Val acc 91.98%  F1 0.918
[Custom] 15/50  Val acc 90.73%  F1 0.906
[Custom] 16/50  Val acc 91.20%  F1 0.910
[Custom] 17/50  Val acc 89.88%  F1 0.898
[Custom] 18/50  Val acc 90.65%  F1 0.904
[Custom] 19/50  Val acc 92.37%  F1 0.921
[Custom] 20/50  Val acc 92.13%  F1 0.919
[Custom] 21/50  Val acc 92.45%  F1 0.923
[Custom] 22/50  Val acc 92.37%  F1 0.922
[Custom] 23/50  Val acc 92.06%  F1 0.919
[Custom] 24/50  Val acc 92.37%  F1 0.922
[Custom] 25/50  

## 5 Model B – ResNet‑18 Transfer Learning (RN18‑TL)

**Two‑phase schedule**

1. **Epoch 0** – train **only** the new `fc` layer (frozen backbone)  
2. **Epoch 1 → 50** – unfreeze entire network, lower LR (`1e‑4`)  


In [38]:

import torchvision.models as models
resnet = models.resnet18(weights='IMAGENET1K_V1')
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
resnet = resnet.to(DEVICE)

# Freeze backbone for 1 epoch
for p in resnet.parameters(): p.requires_grad = False
for p in resnet.fc.parameters(): p.requires_grad = True

crit_tl = nn.CrossEntropyLoss()
opt_tl  = optim.Adam(resnet.fc.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_tl = 0.0
for ep in range(N_EPOCHS):
    if ep == 1:
        for p in resnet.parameters(): p.requires_grad = True
        opt_tl = optim.Adam(resnet.parameters(), lr=LR*0.1, weight_decay=WEIGHT_DECAY)
    train_one_epoch(resnet, train_loader, crit_tl, opt_tl)
    _, v_acc, v_f1, _, _ = evaluate(resnet, val_loader, crit_tl)
    print(f'[ResNet18] {ep+1:02d}/{N_EPOCHS}  Val acc {v_acc*100:5.2f}%  F1 {v_f1:.3f}')
    best_tl = max(best_tl, v_acc)
print(f'Best ResNet‑18 accuracy: {best_tl*100:.2f}%')


[ResNet18] 01/50  Val acc 79.83%  F1 0.795
[ResNet18] 02/50  Val acc 96.26%  F1 0.961
[ResNet18] 03/50  Val acc 93.85%  F1 0.934
[ResNet18] 04/50  Val acc 98.29%  F1 0.982
[ResNet18] 05/50  Val acc 96.34%  F1 0.962
[ResNet18] 06/50  Val acc 98.52%  F1 0.985
[ResNet18] 07/50  Val acc 95.87%  F1 0.957
[ResNet18] 08/50  Val acc 97.90%  F1 0.978
[ResNet18] 09/50  Val acc 98.83%  F1 0.988
[ResNet18] 10/50  Val acc 98.21%  F1 0.982
[ResNet18] 11/50  Val acc 98.29%  F1 0.982
[ResNet18] 12/50  Val acc 97.66%  F1 0.976
[ResNet18] 13/50  Val acc 99.07%  F1 0.990
[ResNet18] 14/50  Val acc 97.66%  F1 0.976
[ResNet18] 15/50  Val acc 98.44%  F1 0.984
[ResNet18] 16/50  Val acc 98.36%  F1 0.983
[ResNet18] 17/50  Val acc 97.74%  F1 0.977
[ResNet18] 18/50  Val acc 98.75%  F1 0.987
[ResNet18] 19/50  Val acc 98.60%  F1 0.985
[ResNet18] 20/50  Val acc 99.22%  F1 0.992
[ResNet18] 21/50  Val acc 97.59%  F1 0.975
[ResNet18] 22/50  Val acc 98.05%  F1 0.981
[ResNet18] 23/50  Val acc 98.83%  F1 0.988
[ResNet18] 

## 4 Model C – LpCNN

| Hyper‑parameter | Value |
|-----------------|-------|
| Input size      | 32 × 32 |
| Batch size      | 32 |
| Epochs          | 50 |
| Optimizer       | Adam (`lr = 1e‑3`, `wd = 1e‑4`) |
| Scheduler       | ReduceLROnPlateau (patience = 10) |


In [ ]:
import os, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# local imports
import importlib
import src.lpconv as lp
importlib.reload(lp)
from src.lpconv import LpCNN, init_lp_weights

from src.dataset import BrainTumorDataset

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT  = r"../cleaned_data"
BATCH      = 32
EPOCHS     = 50
LR         = 1e-3

# same transforms as your friend
train_tf = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3,1,1) if x.shape[0]==1 else x),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])
test_tf = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3,1,1) if x.shape[0]==1 else x),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

train_ds = BrainTumorDataset(DATA_ROOT, "Training", transform=train_tf)
test_ds  = BrainTumorDataset(DATA_ROOT, "Testing",  transform=test_tf)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH)

model = LpCNN(num_classes=len(train_ds.classes)).to(DEVICE)
model.apply(init_lp_weights)  # friend-level init

crit = nn.CrossEntropyLoss()
opt  = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
sched= optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10)

print(f"Classes are: {train_ds.classes}")
best_acc = 0
for ep in range(1, EPOCHS+1):
    # train
    model.train()
    running_loss = 0.0
    for x, y in tqdm(train_dl, desc=f"Epoch {ep}/{EPOCHS}", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        out = model(x)
        loss= crit(out, y)
        loss.backward()
        opt.step()
        running_loss += loss.item()
    running_loss /= len(train_dl)

    # validation
    model.eval()
    val_loss, correct, total= 0.0,0,0
    with torch.no_grad():
        for x,y in test_dl:
            x,y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            val_loss += crit(out,y).item()
            _, pred = out.max(1)
            correct+= (pred==y).sum().item()
            total  += y.size(0)
    val_loss /= len(test_dl)
    acc = correct/total
    sched.step(val_loss)
    best_acc= max(best_acc, acc)
    print(f"[{ep:02d}/{EPOCHS}] train={running_loss:.3f} val={val_loss:.3f}  acc={acc*100:5.2f}%")

print(f"Done. Best accuracy = {best_acc*100:.2f}%")
torch.save(model.state_dict(), "best_lp_cnn_tumor.pth")


Classes are: ['NoTumor', 'Pituitary', 'Meningioma', 'Glioma']


[01/50] train=0.783 val=0.881  acc=66.43%


[02/50] train=0.607 val=0.630  acc=76.40%


[03/50] train=0.561 val=1.286  acc=54.21%


[04/50] train=0.528 val=1.660  acc=48.52%


[05/50] train=0.513 val=0.714  acc=74.53%


... (41 outputs left out)

[47/50] train=0.224 val=0.373  acc=85.20%


[48/50] train=0.224 val=0.789  acc=73.60%


[49/50] train=0.223 val=0.280  acc=89.95%


[50/50] train=0.227 val=0.348  acc=87.46%
Done. Best accuracy = 89.95%
